# Beacon — End-to-end smoke tests

Five real-world tests against a running Beacon server (`make dev`, default port 8003).

1. **Classic question** — direct text claim via `POST /check`
2. **arXiv article** — preprint URL via `POST /check-url`
3. **Doctissimo article** — French health blog via `POST /check-url`
4. **YouTube video** — TED talk with captions via `POST /check-url`
5. **TikTok video** — short health video via `POST /check-url` (requires ffmpeg + Gemma 4 E4B downloaded)

Each cell is independent — run any subset. Long-running cells print progress + the full server response.

In [1]:
import json, time, textwrap
import requests

BASE = "http://localhost:8003"
TIMEOUT = 600  # the agentic pipeline can take 1-2 min for URLs

BAND_EMOJI = {
    "Supported": "✅", "Partially supported": "🟡",
    "Insufficient evidence": "⚪", "Contradicted": "❌",
    "Known misinformation": "🚫",
}

def health():
    r = requests.get(f"{BASE}/healthz", timeout=5)
    print("server:", r.status_code, r.json())

def post_check(payload, label):
    print(f"▶ {label}")
    print(f"  payload keys: {list(payload.keys())}")
    t0 = time.perf_counter()
    r = requests.post(f"{BASE}/check", json=payload, timeout=TIMEOUT)
    elapsed = time.perf_counter() - t0
    print(f"  HTTP {r.status_code} in {elapsed:.1f}s")
    if not r.ok:
        print("  ", r.text[:1500])
        return None
    data = r.json()
    print(f"  results: {len(data['results'])} claim(s) · server elapsed: {data['elapsed_seconds']}s\n")
    for i, item in enumerate(data["results"], 1):
        c, v = item["claim"], item["verdict"]
        e = BAND_EMOJI.get(v.get("band"), "❓")
        print(f"  --- Claim {i} ---")
        print(f"  {e} {v.get('band')} (score {v.get('score')})")
        print(f"  \"{c['text']}\"")
        if v.get("narrative"):
            print(textwrap.indent(textwrap.fill(v["narrative"], 88), "    "))
        sources = []
        for f in v.get("findings", []):
            for s in (f.get("sources") or []):
                sources.append(f"{f['agent']:>10}  {(s.get('title') or '')[:60]}  {s.get('url','')[:60]}")
        if sources:
            print(f"    sources ({len(sources)}):")
            for s in sources[:5]: print(f"    - {s}")
            if len(sources) > 5: print(f"    … and {len(sources) - 5} more")
        print()
    return data

health()

server: 200 {'ok': True}


## Test 1 — Classic question (text claim)

Submits a single hand-typed claim through `POST /check`. Tests the original single-claim flow: `claim_extractor → evidence_fan → combiner → reviewer → narrative`.

In [2]:
post_check(
    {"text": "Drinking lemon water on an empty stomach cures stage IV cancer."},
    label="Test 1 — text claim",
)
print("⏸  cooldown 65s before next test"); time.sleep(65)

▶ Test 1 — text claim
  payload keys: ['text']


  HTTP 200 in 50.6s
  results: 1 claim(s) · server elapsed: 50.55s

  --- Claim 1 ---
  ❌ Contradicted (score -1.0)
  "Drinking lemon water on an empty stomach cures stage IV cancer."
    There is no scientific evidence that lemon water can cure stage IV cancer. The World
    Health Organization states that no food or drink has been shown to cure cancer.
    sources (1):
    -        who  Drinking lemon water cures cancer.  https://www.who.int/news-room/fact-sheets/detail/cancer

⏸  cooldown 65s before next test


## Test 2 — arXiv article

A real biomedical preprint. Tests `trafilatura` extraction + multi-claim split.
Med-PaLM 2 paper from Google Research (high-confidence health-related content).

In [3]:
post_check(
    {"url": "https://arxiv.org/abs/2305.09617"},  # Med-PaLM 2
    label="Test 2 — arXiv article URL",
)
print("⏸  cooldown 65s"); time.sleep(65)

▶ Test 2 — arXiv article URL
  payload keys: ['url']


  HTTP 422 in 21.7s
   {"detail":"No claims produced. The Manager may have decided the input contained no checkable health claims. Manager output: {\n  \"claims\": []\n}"}
⏸  cooldown 65s


## Test 3 — Doctissimo article (French)

Tests `trafilatura` on a French wellness blog + multilingual claim handling (claims should be translated to English for retrieval).

In [4]:
post_check(
    {"url": "https://www.doctissimo.fr/sante/cholesterol"},
    label="Test 3 — Doctissimo article (French)",
)
print("⏸  cooldown 65s"); time.sleep(65)

▶ Test 3 — Doctissimo article (French)
  payload keys: ['url']


  HTTP 200 in 93.7s
  results: 4 claim(s) · server elapsed: 93.74s

  --- Claim 1 ---
  ⚪ Insufficient evidence (score 0.0)
  "Daily napping can increase the risk of heart attacks by 50%."
    Aucune preuve n'a été trouvée dans les bases de données consultées pour confirmer que la
    sieste quotidienne augmente le risque d'infarctus du myocarde de 50 %. Bien que des
    liens existent entre le sommeil et la santé cardiovasculaire, cette statistique précise
    n'est pas étayée par les sources analysées.
    sources (1):
    -     pubmed  Circadian rhythms in cardiovascular disease.  https://pubmed.ncbi.nlm.nih.gov/40663373/

  --- Claim 2 ---
  ❌ Contradicted (score -0.7)
  "Cardiovascular mortality is multiplied by two to four in people suffering from anxiety or depression."
    Une méta-analyse indique que la dépression est associée à une augmentation du risque de
    mortalité cardiovasculaire avec un rapport de risque (HR) de 1,44, ce qui est nettement
    inférieur au multiplicat

## Test 4 — YouTube video (with captions)

Tests `youtube-transcript-api` ingestion. TED talk on sugar and the body — has captions.

In [5]:
post_check(
    {"url": "https://www.youtube.com/watch?v=78CJOrZ1Z2c"},  # TED-Ed
    label="Test 4 — YouTube video (captions)",
)
print("⏸  cooldown 65s"); time.sleep(65)

▶ Test 4 — YouTube video (captions)
  payload keys: ['url']


  HTTP 500 in 5.3s
   Internal Server Error
⏸  cooldown 65s


## Test 5 — TikTok video

Tests the **Gemma 4 E4B audio path**. Requires:
- working `ffmpeg` (`brew reinstall ffmpeg` if dyld errors)
- HuggingFace login + Gemma license accepted
- ~16 GB free disk for the model (downloaded lazily on first run, may take 5-15 min)

Expect this cell to fail until the above are set up. The error should be informative (`ffmpeg not found`, `gated repo`, etc.).

In [6]:
# TikTok requires ffmpeg + Gemma 4 E4B local. Expect failure if either missing.
# Substitute with any current TikTok health video — TikTok URLs rot fast.
post_check(
    {"url": "https://www.tiktok.com/@drsoodmd/video/7407094892503944491"},
    label="Test 5 — TikTok video (audio transcription)",
)

▶ Test 5 — TikTok video (audio transcription)
  payload keys: ['url']


  HTTP 500 in 7.7s
   Internal Server Error
